In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1 - Abrir o arquivo
path = r"h:\3etapa_Py\aula4\CD2022_Populacao_2010_Compatibilizada_20231222 (1).xlsx"

bruto = pd.read_excel(path, header=None)
bruto.head(5)

In [ ]:
# 2 - Limpar: linha 2 é o cabeçalho, dados começam na linha 3
df = bruto.iloc[2:].copy()
df.columns = bruto.iloc[2]
df = df.iloc[1:].reset_index(drop=True)

# Remover coluna vazia (coluna 0)
df = df.dropna(axis=1, how="all")

# Renomear colunas para facilitar
df.columns = ["UF", "COD. UF", "COD. MUNIC", "NOME DO MUNICÍPIO", "pop_2010_sinopse", "pop_2010", "pop_2022"]

# Garantir que as colunas numéricas são números
for col in ["pop_2010", "pop_2022"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()

In [ ]:
# 3 - Explorar o DataFrame
print(df.shape)
df.dtypes

In [ ]:
# 4 - Estatísticas básicas
df[["pop_2010", "pop_2022"]].describe().round(0)

## Agrupando por Estado (UF)

In [ ]:
# 5 - GROUP BY: população total por estado
pop_estado = (
    df.groupby(["COD. UF", "UF"], as_index=False)
      .agg(
          pop_2010=("pop_2010", "sum"),
          pop_2022=("pop_2022", "sum"),
      )
)

# Calcular crescimento absoluto e percentual
pop_estado["crescimento"] = pop_estado["pop_2022"] - pop_estado["pop_2010"]
pop_estado["crescimento_pct"] = (pop_estado["crescimento"] / pop_estado["pop_2010"] * 100).round(2)

pop_estado = pop_estado.sort_values("crescimento", ascending=False).reset_index(drop=True)
pop_estado

In [ ]:
# 6 - Salvar tabela por estado em CSV
pop_estado.to_csv(r"h:\3etapa_Py\aula4\populacao_por_estado.csv", sep=";", index=False)
print("Salvo!")

## Agrupando por Município

In [ ]:
# 7 - Trabalhar por município: calcular crescimento
pop_municipio = df[["UF", "COD. UF", "COD. MUNIC", "NOME DO MUNICÍPIO", "pop_2010", "pop_2022"]].copy()

pop_municipio["crescimento"] = pop_municipio["pop_2022"] - pop_municipio["pop_2010"]
pop_municipio["crescimento_pct"] = (pop_municipio["crescimento"] / pop_municipio["pop_2010"] * 100).round(2)

# 8 - Ordenar por maior crescimento absoluto
pop_municipio = pop_municipio.sort_values("crescimento", ascending=False).reset_index(drop=True)
pop_municipio.head(10)

In [ ]:
# 9 - Salvar tabela por município em CSV
pop_municipio.to_csv(r"h:\3etapa_Py\aula4\populacao_por_municipio.csv", sep=";", index=False)
print("Salvo!")

## Gráficos

In [ ]:
# 10 - Gráfico: 10 UFs com maior crescimento absoluto (2010 vs 2022)
top = pop_estado.head(10).melt(
    id_vars="UF",
    value_vars=["pop_2010", "pop_2022"],
    var_name="ano",
    value_name="populacao",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top, x="UF", y="populacao", hue="ano", ax=ax)
ax.set_title("10 UFs com maior crescimento absoluto: 2010 vs 2022")
ax.set_ylabel("Habitantes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 11 - Gráfico: crescimento percentual por estado
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=pop_estado.sort_values("crescimento_pct", ascending=False),
    x="UF", y="crescimento_pct", ax=ax
)
ax.set_title("Crescimento populacional % por UF (2010–2022)")
ax.set_ylabel("Crescimento (%)")
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()